<div dir="rtl" style="text-align:right"><h1>שיעור 3, חלק 2: איך מודל שפה כותב – שלב F: תשובה רהוטה עדיין דורשת ראיות</h1>
<p><b>המחברת הזו היא לשלב F («תשובה רהוטה עדיין דורשת ראיות») בשיעור באתר הקורס – ורק לו.</b> את ההסבר קוראים באתר; כאן מריצים ומשנים.</p>
<p>מריצים תא אחרי תא בלחיצה על ▶, מלמעלה למטה. מעבד רגיל (<code>CPU</code>) מספיק; אין צורך ב-<code>GPU</code>, במפתח <code>API</code> או במנוי. <b>אין חובת הגשת מחברת.</b> תא ההכנה מוריד מודל פתוח של כ-1GB; ההורדה לוקחת בדרך כלל דקה או שתיים. אם מופיעה אזהרה על <code>HF_TOKEN</code> – אפשר להתעלם ממנה: המודל ציבורי.</p></div>

In [ ]:
# Setup: transformers and torch are already installed on Google Colab.
# Loads the small open model Qwen/Qwen2.5-0.5B (about 1GB; usually a minute or two).
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# "eager" attention lets the model return its attention weights; the predictions are the same.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32, attn_implementation="eager")
model.eval()
print(f"Loaded {MODEL_NAME}: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} billion parameters")

<div dir="rtl" style="text-align:right"><h2>4 | השלמה שוטפת אינה עובדה</h2>
<p>ההודעה «מיכל רוצה שלוש עוגיות למחר» לא כוללת שעת איסוף. מה יקרה אם נבקש מהמודל להמשיך את הטקסט אחרי «שעת האיסוף:»? התא הבא מגריל שלוש השלמות (בכל פעם בוחרים טוקן לפי ההסתברויות, לכן התוצאות משתנות – וגם אצלכם עשויות לצאת אחרות).</p><blockquote>① נחשו לפני ההרצה: המודל יכתוב «לא ידוע», או ימציא שעה?</blockquote></div>

In [ ]:
prompt = "מיכל רוצה שלוש עוגיות למחר. שעת האיסוף:"
ids = tokenizer(prompt, return_tensors="pt").input_ids
for seed in range(3):
    torch.manual_seed(seed)
    out = model.generate(ids, max_new_tokens=8, do_sample=True, temperature=0.8, pad_token_id=tokenizer.eos_token_id)
    print(f"Sample {seed + 1}: {prompt}" + tokenizer.decode(out[0, ids.shape[1]:]).split("\n")[0])

<div dir="rtl" style="text-align:right"><p>המודל משלים שעה שנשמעת סבירה – כי זה מה שמודל שפה עושה: ממשיך טקסט באופן סביר. <b>אבל השעה לא הופיעה בהודעה</b>, ולכן כל שעה כזו היא המצאה. גם צ׳טבוט גדול יכול לעשות את אותו הדבר בניסוח בטוח מאוד.</p>
<h3>4א | בדיקה מבוססת קלט</h3>
<p>בשלב F באתר («תשובה רהוטה עדיין דורשת ראיות») תבקשו מהצ׳טבוט לסכם את ההזמנה בפורמט <code>JSON</code> – טקסט מסודר של שמות שדות וערכים. התא הבא בודק את הפלט מול ההודעה המקורית בשתי בדיקות:</p>
<ol>
<li>הכמות בפלט שווה לכמות בהודעה (3).</li>
<li>שעת האיסוף בפלט ריקה (<code>null</code>) ומסומנת כחסרה – כי היא לא נמסרה.</li>
</ol>
<p>הפלט שכבר נמצא בתא <b>נכתב ביד כדוגמה</b>; הוא לא פלט של צ׳טבוט. החליפו אותו בפלט שקיבלתם (מה שבין שלושת המירכאות) והריצו שוב (③ – זה הדבר האחד שמשנים כאן).</p>
<p>בגרסה שבה מחקתם את הכמות מההודעה, הבדיקה הראשונה <b>צפויה להיכשל</b> (היא מחפשת 3). שם בדקו בעין: הכמות צריכה להיות <code>null</code> ולהופיע ב-<code>missing</code>.</p></div>

In [ ]:
import json

# Replace the text between the triple quotes with your chatbot's output.
# The example below was written by hand; it is NOT a chatbot's output.
chatbot_output = """
{"customer": "מיכל", "product": "עוגיות", "quantity": 3, "pickup_day": "מחר", "pickup_time": null, "missing": ["pickup_time"]}
"""

text = chatbot_output.strip()
if text.startswith("```"):  # chatbots often wrap JSON in a code fence
    text = text.strip("`").removeprefix("json").strip()
try:
    order = json.loads(text)
except json.JSONDecodeError as error:
    order = None
    print("This is not valid JSON:", error)
    print("Ask the chatbot to answer with the JSON only, then paste it again.")

if order is not None:
    checks = {
        "quantity is 3, as in the message": order.get("quantity") == 3,
        "pickup_time is empty (null)": order.get("pickup_time") is None,
        "pickup_time is listed as missing": "pickup_time" in (order.get("missing") or []),
    }
    for name, passed in checks.items():
        print(("PASS  " if passed else "FAIL  ") + name)

<div dir="rtl" style="text-align:right"><p><b>סיימתם את המחברת של שלב F.</b> חזרו ללשונית של אתר הקורס וענו שם על שאלות השלב.</p></div>